# Python Bindings Development Roadmap

This notebook documents what the Rust Barter engine exposes vs. what the Python
bindings currently cover, and lays out a phased implementation plan to close the
gaps.

## Current State

| Feature Area | Python Status | Rust Status |
|---|---|---|
| Instruments & Indexing | **Complete** | Complete |
| Market Data Streaming (trades) | **Complete** (4 exchanges) | Complete (12+ exchanges) |
| Market Data (L1/L2 books) | Read-only fields on `MarketEvent` | Full `OrderBookMap` + manager |
| Trading Statistics | **Complete** | Complete |
| Engine / Backtesting | Partial (`run_backtest()` with noop strategy) | Full lifecycle |
| Custom Strategies | **Not exposed** (`PyStrategy` stub exists) | 4 traits, multi-strategy composition |
| Custom Risk Management | **Not exposed** (`PyRiskManager` stub exists) | Full `RiskManager` trait |
| Order Management | **Not exposed** | `OrderRequestOpen`, `OrderRequestCancel` |
| Position Tracking | **Not exposed** | `PositionManager`, `PositionExited` |
| Live Trading | **Not exposed** | Full `ExecutionClient` trait |
| Account Events | **Not exposed** | `AccountEvent`, `AccountEventKind` |

---
## Phase 1: Strategy Callbacks (Priority: Critical)

**Goal**: Let users write trading strategies in Python that run inside the Rust engine.

**Why first**: This is the single most-requested feature. Without it, the Python
package is limited to data streaming and post-hoc statistics.

### Tasks

1. **Expose `EngineState` read accessors to Python**
   - `barter-python/src/engine.rs` or new `state.rs`
   - Wrap key state fields: current positions, balances, latest prices, open orders
   - Read-only — the strategy should not mutate state directly
   - Consider a `PyEngineStateSnapshot` that copies data into Python-friendly types
     on each tick (avoids holding Rust borrows across the FFI boundary)

2. **Expose order request types**
   - `PyOrderRequestOpen(exchange, instrument, side, price, quantity, kind, strategy_id)`
   - `PyOrderRequestCancel(exchange, instrument, order_id)`
   - These are the return values from a Python strategy callback

3. **Implement `AlgoStrategy` via Python callback**
   - `barter-python/src/strategy.rs` — replace the noop stub
   - Accept a Python callable: `fn(state: PyEngineStateSnapshot) -> (List[Cancel], List[Open])`
   - Use `pyo3::Python::with_gil()` to call back into Python on each engine tick
   - Performance note: GIL acquisition per tick is acceptable for backtesting;
     for live trading, consider batching or async callbacks

4. **Implement `ClosePositionsStrategy` via Python callback**
   - Optional second callable: `fn(state, filter) -> (List[Cancel], List[Open])`
   - Default: generate IOC market orders to flatten all positions
     (use `build_ioc_market_order_to_close_position` from Rust)

5. **Wire into `run_backtest()` and add `run_engine()`**
   - `run_backtest(config, data, strategy_fn, risk_fn=None)` — add strategy parameter
   - New `run_engine(config, strategy_fn, risk_fn=None)` for live streams

### Files to modify
- `barter-python/src/strategy.rs` — core implementation
- `barter-python/src/engine.rs` — wire strategy into engine builder
- `barter-python/src/instrument.rs` — may need `InstrumentIndex` exposed
- New: `barter-python/src/order.rs` — order request types
- New: `barter-python/src/state.rs` — engine state snapshot

### Definition of Done
- User can write `def my_strategy(state): return ([], [open_order])` in Python
- Pass it to `run_backtest()` and see trades in the resulting `TradingSummary`
- Notebook `02_engine_and_backtesting.ipynb` updated with working strategy example

---
## Phase 2: Risk Management Callbacks (Priority: High)

**Goal**: Let users implement custom risk checks in Python.

### Tasks

1. **Expose risk utility types**
   - `PyRiskApproved`, `PyRiskRefused` wrappers (or just filter in Rust)
   - `calculate_quote_notional()` and `calculate_abs_percent_difference()` as
     standalone Python functions

2. **Implement `RiskManager` via Python callback**
   - `barter-python/src/risk.rs` — replace the approve-all stub
   - Accept a callable: `fn(state, cancels, opens) -> (approved_cancels, approved_opens, refused_opens)`
   - Default: approve all (current behaviour)

3. **Wire into `run_backtest()`**
   - `run_backtest(..., risk_fn=my_risk_check)`

### Files to modify
- `barter-python/src/risk.rs` — core implementation
- `barter-python/src/engine.rs` — wire risk manager into engine builder

### Definition of Done
- User can write a risk function that rejects orders above a notional threshold
- Notebook `02_engine_and_backtesting.ipynb` updated with working risk example

---
## Phase 3: Position & Account Event Exposure (Priority: Medium)

**Goal**: Give Python access to live position tracking and account events.

### Tasks

1. **Expose `PositionExited` and `PositionManager`**
   - `PyPositionExited` with fields: instrument, side, entry price, quantity, PnL,
     fees, enter/exit times
   - Read-only snapshot from engine state

2. **Expose `AccountEvent` and `AccountEventKind`**
   - `PyAccountEvent` wrapping trade fills, balance updates, order status changes
   - Needed for the strategy callback to observe its own fills

3. **Add event callback hooks**
   - `on_trade(fn)`, `on_balance_update(fn)`, `on_position_closed(fn)`
   - Optional callbacks registered on the engine for real-time observation

### Files to modify
- New: `barter-python/src/position.rs`
- New: `barter-python/src/account.rs`
- `barter-python/src/state.rs` — include positions in snapshot

### Definition of Done
- Strategy callback receives current open positions in `state`
- Closed positions are accessible during and after the backtest

---
## Phase 4: Extended Market Data (Priority: Medium)

**Goal**: Expose L1/L2 order books and additional data kinds to Python.

### Tasks

1. **Expose `OrderBooksL1` subscription kind**
   - `Subscription("binance_spot", "btc", "usdt", data_kind="order_book_l1")`
   - `MarketEvent.best_bid_price` / `best_ask_price` already exist but
     the subscription routing doesn't support it yet

2. **Expose `OrderBooksL2` with `OrderBookMap`**
   - `PyOrderBook` with `bids` and `asks` as list-of-tuples `[(price, qty), ...]`
   - `PyOrderBookMap` with `.snapshot(instrument)` method
   - Background update manager (as in Rust notebook)

3. **Add remaining exchanges**
   - Currently: Binance Spot, Binance Futures, OKX, Coinbase
   - Add: Kraken, Bybit, Bitfinex, Gateio, Bitmex
   - Each exchange just needs a match arm in `data.rs`

### Files to modify
- `barter-python/src/data.rs` — subscription routing, new data kinds, new exchanges
- New: `barter-python/src/orderbook.rs` — order book types

### Definition of Done
- L1 and L2 book data available as Python objects from `build_market_stream()`
- All 9+ Rust-supported exchanges accessible from Python

---
## Phase 5: Live Trading (Priority: Lower)

**Goal**: Run the full engine with real exchange connections.

### Tasks

1. **Expose `SystemBuilder` with live clock and execution**
   - `run_live(config, strategy_fn, risk_fn)` — uses `LiveClock`, real WebSockets,
     and live execution clients
   - Config switches from `mocked_exchange` to real exchange credentials

2. **Expose engine control commands**
   - `system.enable_trading()` / `system.disable_trading()`
   - `system.cancel_orders(filter)` / `system.close_positions(filter)`
   - `system.shutdown()` → `TradingSummary`

3. **Expose `OnDisconnectStrategy` and `OnTradingDisabled`**
   - Optional callbacks: `on_disconnect(exchange)`, `on_trading_disabled()`
   - Defaults: cancel all orders on disconnect, no-op on disable

4. **Add audit stream exposure**
   - `system.audit_stream()` → async iterator of engine events
   - Useful for logging, monitoring, and debugging

### Files to modify
- `barter-python/src/engine.rs` — add `run_live()`, expose `System` wrapper
- `barter-python/src/strategy.rs` — add disconnect/disabled callbacks

### Definition of Done
- User can run a Python strategy against live exchange data
- Graceful shutdown returns a `TradingSummary`

---
## Implementation Priority Matrix

```
Impact on Users
     ▲
     │
High │  Phase 1: Strategy     Phase 5: Live Trading
     │  Callbacks              (needs Phase 1-3)
     │
     │  Phase 2: Risk          Phase 4: Extended
Med  │  Callbacks              Market Data
     │
     │  Phase 3: Position
Low  │  & Account Events
     │
     └──────────────────────────────────────────▶
          Low          Med          High
                  Implementation Effort
```

### Recommended Order: 1 → 2 → 3 → 4 → 5

Phases 1 and 2 are the highest-leverage work — they unlock custom backtesting,
which is the primary use case. Phase 3 enriches the strategy authoring
experience. Phase 4 broadens data coverage. Phase 5 is the final step to
production readiness.

---
## Technical Notes

### GIL Considerations

Calling Python callbacks from the Rust engine requires acquiring the GIL on
each engine tick. For backtesting this is acceptable (the engine is
single-threaded and I/O-free). For live trading, consider:

- Releasing the GIL during Rust-only processing (`py.allow_threads()`)
- Batching multiple events before calling back into Python
- Using `tokio::spawn_blocking()` for the Python callback to avoid blocking
  the async runtime

### State Snapshot Pattern

The recommended pattern for strategy callbacks is a **snapshot**: copy the
relevant engine state into Python-owned objects before calling the callback.
This avoids holding Rust borrows across the FFI boundary and simplifies
lifetime management.

```
Engine tick → copy state to PyEngineStateSnapshot → acquire GIL → call Python
           → collect returned orders → release GIL → feed orders to risk manager
```

### Existing Stubs

The files `barter-python/src/strategy.rs` and `barter-python/src/risk.rs`
already contain stub implementations (`PyStrategy` and `PyRiskManager`) that
implement the required Rust traits. The work is to:

1. Add a `PyObject` callback field to each struct
2. In the trait method, acquire the GIL and call the Python function
3. Convert the Python return value back to Rust order types
4. Handle Python exceptions as `RiskRefused` or engine errors